### Multi-Label Enzyme-Substrate classification

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("train.csv")
print(df.head())
df.count()

   id     BertzCT       Chi1     Chi1n      Chi1v     Chi2n      Chi2v  \
0   0  323.390782   9.879918  5.875576   5.875576  4.304757   4.304757   
1   1  273.723798   7.259037  4.441467   5.834958  3.285046   4.485235   
2   2  521.643822  10.911303  8.527859  11.050864  6.665291   9.519706   
3   3  567.431166  12.453343  7.089119  12.833709  6.478023  10.978151   
4   4  112.770735   4.414719  2.866236   2.866236  1.875634   1.875634   

      Chi3v     Chi4n  EState_VSA1  ...  SlogP_VSA3  VSA_EState9  fr_COO  \
0  2.754513  1.749203     0.000000  ...    4.794537    35.527357       0   
1  2.201375  1.289775    45.135471  ...   13.825658    44.707310       0   
2  5.824822  1.770579    15.645394  ...   17.964475    45.660120       0   
3  7.914542  3.067181    95.639554  ...   31.961948    87.509997       0   
4  1.036450  0.727664    17.980451  ...    9.589074    33.333333       2   

   fr_COO2  EC1  EC2  EC3  EC4  EC5  EC6  
0        0    1    1    0    0    0    0  
1        0  

,0
id,14838
BertzCT,14838
Chi1,14838
Chi1n,14838
Chi1v,14838
Chi2n,14838
Chi2v,14838
Chi3v,14838
Chi4n,14838
EState_VSA1,14838


### Data Preparation for Multi-Label Classification


In [15]:
X = df.drop(columns=['id', 'EC1', 'EC2', 'EC3', 'EC4', 'EC5', 'EC6'])
y = df[['EC1', 'EC2']]

In [16]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (11870, 31)
X_test shape: (2968, 31)
y_train shape: (11870, 2)
y_test shape: (2968, 2)


In [17]:
# Removed stratify=y to avoid the error. Please consider the options mentioned above for a more robust solution.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (11870, 31)
X_test shape: (2968, 31)
y_train shape: (11870, 2)
y_test shape: (2968, 2)


### Decision Tree Classifier


In [19]:
from sklearn.tree import DecisionTreeClassifier


dt_classifier = DecisionTreeClassifier(random_state=42)

dt_classifier.fit(X_train, y_train)

print("Decision Tree Classifier trained")

Decision Tree Classifier trained


### Evaluate Decision Tree Classifier

 For multi-label classification, common metrics include Jaccard score, Hamming loss, and F1-score (micro and macro averages).

In [20]:
from sklearn.metrics import jaccard_score, hamming_loss, f1_score, roc_auc_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    # Multi-label metrics
    jaccard = jaccard_score(y_test, y_pred, average='samples')
    hamming = hamming_loss(y_test, y_pred)
    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')

    print(f"Classifier Evaluation:")
    print(f"  Jaccard Score (samples): {jaccard:.4f}")
    print(f"  Hamming Loss: {hamming:.4f}")
    print(f"  F1 Score (Micro): {f1_micro:.4f}")
    print(f"  F1 Score (Macro): {f1_macro:.4f}")

    roc_aucs_per_ec = {}
    y_pred_proba_list = model.predict_proba(X_test)
    if not isinstance(y_pred_proba_list, list):
      y_pred_proba_list = [y_pred_proba_list]

    print("  ROC-AUC per EC label:")
    for i, col in enumerate(y_test.columns):
        roc_auc = roc_auc_score(y_test[col], y_pred_proba_list[i][:, 1])
        roc_aucs_per_ec[col] = roc_auc
        print(f"    {col}: {roc_auc:.4f}")

    return jaccard, hamming, f1_micro, f1_macro, roc_aucs_per_ec

res = evaluate_model(dt_classifier, X_test, y_test)
jaccard_dt, hamming_dt, f1_micro_dt, f1_macro_dt, roc_aucs_dt = res

Classifier Evaluation:
  Jaccard Score (samples): 0.6201
  Hamming Loss: 0.3598
  F1 Score (Micro): 0.7535
  F1 Score (Macro): 0.7503
  ROC-AUC per EC label:
    EC1: 0.5665
    EC2: 0.4937


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Random Forest Classifier


In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(random_state=42)

rf_classifier.fit(X_train, y_train)

print("Random Forest Classifier trained")

Random Forest Classifier trained


### Evaluate Random Forest Classifier


In [22]:
res = evaluate_model(rf_classifier, X_test, y_test)
jaccard_rf, hamming_rf, f1_micro_rf, f1_macro_rf, roc_aucs_rf = res

Classifier Evaluation:
  Jaccard Score (samples): 0.7453
  Hamming Loss: 0.2490
  F1 Score (Micro): 0.8474
  F1 Score (Macro): 0.8421
  ROC-AUC per EC label:
    EC1: 0.6944
    EC2: 0.5408


### Gradient Boosting Machine (GBM)

In [23]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

gbm_classifier = MultiOutputClassifier(GradientBoostingClassifier(random_state=42))

gbm_classifier.fit(X_train, y_train)

print("Gradient Boosting Machine trained")

Gradient Boosting Machine trained


### Evaluate Gradient Boosting Machine

In [24]:
res = evaluate_model(gbm_classifier, X_test, y_test)
jaccard_gbm, hamming_gbm, f1_micro_gbm, f1_macro_gbm, roc_aucs_gbm = res

Classifier Evaluation:
  Jaccard Score (samples): 0.7480
  Hamming Loss: 0.2473
  F1 Score (Micro): 0.8490
  F1 Score (Macro): 0.8439
  ROC-AUC per EC label:
    EC1: 0.7016
    EC2: 0.5869


### AdaBoost classifier

In [25]:
from sklearn.ensemble import AdaBoostClassifier

ada_classifier = MultiOutputClassifier(AdaBoostClassifier(random_state=42))

ada_classifier.fit(X_train, y_train)

print("AdaBoost trained")

AdaBoost trained


### Evaluate AdaBoost Classifier

In [26]:
res = evaluate_model(ada_classifier, X_test, y_test)
jaccard_ada, hamming_ada, f1_micro_ada, f1_macro_ada, roc_aucs_ada = res

Classifier Evaluation:
  Jaccard Score (samples): 0.7493
  Hamming Loss: 0.2468
  F1 Score (Micro): 0.8492
  F1 Score (Macro): 0.8439
  ROC-AUC per EC label:
    EC1: 0.6938
    EC2: 0.5811


### Comparison of Models

In [27]:
import numpy as np

def print_results(jaccard, hamming, f1_micro, f1_macro, roc_aucs):
  print(f"  Jaccard Score (samples): {jaccard:.4f}")
  print(f"  Hamming Loss: {hamming:.4f}")
  print(f"  F1 Score (Micro): {f1_micro:.4f}")
  print(f"  F1 Score (Macro): {f1_macro:.4f}")
  print(f"  Average ROC-AUC: {np.mean(list(roc_aucs.values())):.4f}")
  print(f"  ROC-AUC per EC label:")
  for ec_label, auc_score in roc_aucs.items():
      print(f"{ec_label}: {auc_score:.4f}")

print(f"\nDEcision Tree Classifier:")
print_results(jaccard_dt, hamming_dt, f1_micro_dt, f1_macro_dt, roc_aucs_dt)
print(f"\nRandom Forest Classifier:")
print_results(jaccard_rf, hamming_rf, f1_micro_rf, f1_macro_rf, roc_aucs_rf)
print(f"\nGBM Classifier:")
print_results(jaccard_gbm, hamming_gbm, f1_micro_gbm, f1_macro_dt, roc_aucs_gbm)
print(f"\nAdaBoost Classifier:")
print_results(jaccard_ada, hamming_ada, f1_micro_ada, f1_macro_ada, roc_aucs_ada)


DEcision Tree Classifier:
  Jaccard Score (samples): 0.6201
  Hamming Loss: 0.3598
  F1 Score (Micro): 0.7535
  F1 Score (Macro): 0.7503
  Average ROC-AUC: 0.5301
  ROC-AUC per EC label:
EC1: 0.5665
EC2: 0.4937

Random Forest Classifier:
  Jaccard Score (samples): 0.7453
  Hamming Loss: 0.2490
  F1 Score (Micro): 0.8474
  F1 Score (Macro): 0.8421
  Average ROC-AUC: 0.6176
  ROC-AUC per EC label:
EC1: 0.6944
EC2: 0.5408

GBM Classifier:
  Jaccard Score (samples): 0.7480
  Hamming Loss: 0.2473
  F1 Score (Micro): 0.8490
  F1 Score (Macro): 0.7503
  Average ROC-AUC: 0.6442
  ROC-AUC per EC label:
EC1: 0.7016
EC2: 0.5869

AdaBoost Classifier:
  Jaccard Score (samples): 0.7493
  Hamming Loss: 0.2468
  F1 Score (Micro): 0.8492
  F1 Score (Macro): 0.8439
  Average ROC-AUC: 0.6374
  ROC-AUC per EC label:
EC1: 0.6938
EC2: 0.5811


GBM performed best in terms of average ROC-AUC (0.64), Jaccard score (0.75), and F1-score micro (0.849). F1-score micro is a global score counting all TPs, FPs, and FNs across all six classes and then applying the F1 formula. F1-score macro is calculated independently for each class and then the average of these F1 scores is taken. In our case, the micro is more relevant since enzymes can target more than one class at once. Since the competition challenge is based on ROC-AUC evaluation, I would have reported the GBM model results.